# Lab 5: Sequence Labeling
605.646 Natural Language Processing

Fall 2025

This is a two-week lab.

## Background

This laboratory explores sequence labeling using a neural sequence tagger. By way of background, you might optionally try the following:
1.	Work	through	this [introductory PyTorch	tutorial](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html).
2.	Work	through	 this	[RNN tutorial](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html),	which	has	you	 train	an	RNN	 to	classify surnames based	on their	language/ethnicity.

  What	are	the	top	predictions	for	the	name	McNamee?		How	about	Ocasio-Cortez?
3.	We will be using the FLAIR LSTM-CRF system. Skim	through	[this	paper](https://aclanthology.org/C18-1139/) for details on how the system works.

## Setting Up and Running the Tagger

This notebook describes how to run the FLAIR LSTM/CRF tagger on Colab. You will be using your Google drive to store your project code and data. In the following, most of the manipulation of files on your drive are done through Colab. However, it's fine to use the Google Drive interface if you prefer. You may also use a platform other than Colab if you prefer. If you use Colab, you will likely need to pay for a Colab Pro subscription to avoid timeouts (it costs far less than a textbook and can be canceled whenever you want).


This lab needs to run on a GPU. Click the 'runtime' menu above, select 'change runtime type' and verify you're using a GPU.

First, install the FLAIR and gensim packages. This can take as long as three minutes depending on your machinery.

In [1]:
!pip install flair --quiet

You might see version errors, but they do not appear to affect the tagger's operation.

Import various modules.

In [2]:
import flair
from flair.embeddings import WordEmbeddings, FlairEmbeddings, StackedEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer
from flair.data import Corpus
from flair.datasets import ColumnCorpus
from pathlib import Path

Mount your Google drive. You will be prompted to sign in to your Google account.

In [3]:
from google.colab import drive
gdrive_mount = '/content/drive'
drive.mount(gdrive_mount, force_remount=True)

Mounted at /content/drive


Create a directory to house your project, and move to it. The ! spawns a new process, while the % executes the command in the current process. Note that we need to put quotes around the directory name to run this in a Notebook because in their infinite wisdom Google has placed a space in the default gdrive path.

In [4]:
# Change this path if you'd like to work on your lab in a different directory:
labdir = '/content/drive/My Drive/Teaching/605.646/Fall 2025/lab05'
!mkdir -p "$labdir"
%cd "$labdir"

/content/drive/My Drive/Teaching/605.646/Fall 2025/lab05


Now, outside of Colab, obtain the file ```conll03-englishversion.gz``` from Canvas, ungzip it, and place the resulting directory in your $labdir. Notice that the data is broken into training, dev (called ```valid.txt``` here) and testing partitions.

In [5]:
datadir = f"{labdir}/conll03-englishversion"
!ls "$datadir"

metadata  test.txt  train.txt  valid.txt


## Tagger Training

We're now ready to train a tagger. We'll start by naming our run.

In [6]:
runname = 'flair-ner-conll-eng3'

Next, we'll marshal the CoNLL 2003 English named entity recognition data.

In [7]:
# 1. Create the corpus
columns = {0: 'text', 1: 'pos', 2: 'chunk', 3: 'ner'}
corpus = ColumnCorpus(datadir, columns, train_file="train.txt", dev_file="valid.txt", test_file="test.txt")
print(corpus)

2025-10-06 04:35:38,265 Reading data from /content/drive/My Drive/Teaching/605.646/Fall 2025/lab05/conll03-englishversion
2025-10-06 04:35:38,267 Train: /content/drive/My Drive/Teaching/605.646/Fall 2025/lab05/conll03-englishversion/train.txt
2025-10-06 04:35:38,267 Dev: /content/drive/My Drive/Teaching/605.646/Fall 2025/lab05/conll03-englishversion/valid.txt
2025-10-06 04:35:38,268 Test: /content/drive/My Drive/Teaching/605.646/Fall 2025/lab05/conll03-englishversion/test.txt
Corpus: 14987 train + 3466 dev + 3684 test sentences


In [8]:
# 2. Select the label to predict
label_type = 'ner'

In [9]:

# 3. Create a label dictionary from the corpus
label_dict = corpus.make_label_dictionary(label_type=label_type, add_unk=False)
print(label_dict)

2025-10-06 04:35:48,308 Computing label dictionary. Progress:


1it [00:00, 1713.36it/s]
14987it [00:00, 41899.79it/s]

2025-10-06 04:35:48,671 Dictionary created for label 'ner' with 4 values: LOC (seen 7140 times), PER (seen 6600 times), ORG (seen 6321 times), MISC (seen 3438 times)
Dictionary with 4 tags: LOC, PER, ORG, MISC


The tagger embeds its input using the combination of several models.

In [10]:
# 4. Initialize embedding stack with Flair and GloVe
embedding_types = [
    WordEmbeddings('glove'),
    FlairEmbeddings('news-forward'),
    FlairEmbeddings('news-backward'),
]
embeddings = StackedEmbeddings(embeddings=embedding_types)


Now we'll set up a trainer for our model.

In [11]:
# 5. Initialize the sequence tagger
tagger = SequenceTagger(hidden_size=256,
                        embeddings=embeddings,
                        tag_dictionary=label_dict,
                        tag_type=label_type)


2025-10-06 04:35:52,427 SequenceTagger predicts: Dictionary with 17 tags: O, S-LOC, B-LOC, E-LOC, I-LOC, S-PER, B-PER, E-PER, I-PER, S-ORG, B-ORG, E-ORG, I-ORG, S-MISC, B-MISC, E-MISC, I-MISC


In [12]:
# 6. Initialize the trainer
trainer = ModelTrainer(tagger, corpus)

We're ready to train models using the ```train``` method. We can set a maximum number of epochs here. The model will fully train in fifty to sixty epochs. For most of your experoments, you will want to ensure that you run for the same number of epochs. Twenty epochs should be sufficient for most of your experiments.

You'll see at the start of the output a description of the tagger, indicating that it is a Bi-LSTM with a CRF on top and inputs supplied by a combination of embeddings.

In [13]:
# 7. Train the model
trainer.train(f'resources/taggers/{runname}',
              learning_rate=0.1,
              mini_batch_size=32,
              save_model_each_k_epochs=1,
              monitor_test=True,
              max_epochs=3)

2025-10-06 04:35:52,668 ----------------------------------------------------------------------------------------------------
2025-10-06 04:35:52,669 Model: "SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      'glove'
      (embedding): Embedding(400001, 100)
    )
    (list_embedding_1): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
    (list_embedding_2): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=4196, out_features=4196, bias=True)
  (rnn): LSTM(4196, 256, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=512, out_features=19, bias=True)
  (loss_f

/usr/local/lib/python3.12/dist-packages/flair/trainers/trainer.py:107: UserWarning: There should be no best model saved at epoch 1 except there is a model from previous trainings in your training folder. All previous best models will be deleted.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/flair/trainers/trainer.py:545: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


2025-10-06 04:35:58,859 epoch 1 - iter 46/469 - loss 0.71630521 - time (sec): 6.17 - samples/sec: 3270.29 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:04,654 epoch 1 - iter 92/469 - loss 0.51962568 - time (sec): 11.97 - samples/sec: 3375.85 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:10,379 epoch 1 - iter 138/469 - loss 0.43040618 - time (sec): 17.69 - samples/sec: 3375.93 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:16,165 epoch 1 - iter 184/469 - loss 0.36626136 - time (sec): 23.48 - samples/sec: 3430.30 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:22,091 epoch 1 - iter 230/469 - loss 0.32442610 - time (sec): 29.41 - samples/sec: 3441.28 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:27,766 epoch 1 - iter 276/469 - loss 0.29756024 - time (sec): 35.08 - samples/sec: 3430.53 - lr: 0.100000 - momentum: 0.000000
2025-10-06 04:36:33,487 epoch 1 - iter 322/469 - loss 0.27524429 - time (sec): 40.80 - samples/sec: 3448.51 - lr: 0.100000 - momentum: 0.00

100%|██████████| 55/55 [00:15<00:00,  3.60it/s]

2025-10-06 04:37:10,993 DEV : loss 0.06633012741804123 - f1-score (micro avg)  0.9019


...

2025-10-06 04:39:58,739 TEST : loss 0.06106720119714737 - f1-score (micro avg)  0.9034


2025-10-06 04:39:58,920  - 0 epochs without improvement
2025-10-06 04:39:58,925 saving best model
2025-10-06 04:40:01,783 ----------------------------------------------------------------------------------------------------
2025-10-06 04:40:01,819 Loading model from best epoch ...
2025-10-06 04:40:03,365 SequenceTagger predicts: Dictionary with 19 tags: O, S-LOC, B-LOC, E-LOC, I-LOC, S-PER, B-PER, E-PER, I-PER, S-ORG, B-ORG, E-ORG, I-ORG, S-MISC, B-MISC, E-MISC, I-MISC, <START>, <STOP>


100%|██████████| 58/58 [00:11<00:00,  5.02it/s]


2025-10-06 04:40:16,081 
Results:
- F-score (micro) 0.9034
- F-score (macro) 0.8891
- Accuracy 0.8585

By class:
              precision    recall  f1-score   support

         ORG     0.8619    0.8796    0.8707      1661
         LOC     0.9284    0.9101    0.9192      1668
         PER     0.9682    0.9610    0.9646      1617
        MISC     0.8178    0.7863    0.8017       702

   micro avg     0.9064    0.9003    0.9034      5648
   macro avg     0.8941    0.8843    0.8891      5648
weighted avg     0.9065    0.9003    0.9033      5648

2025-10-06 04:40:16,082 ----------------------------------------------------------------------------------------------------


{'test_score': 0.9033576123645408}

Here are the files that we've built:

In [14]:
!pwd
!ls -l resources/taggers/$runname

/content/drive/MyDrive/Teaching/605.646/Fall 2025/lab05
total 2023320
-rw------- 1 root root 414160181 Oct  6 04:40 best-model.pt
-rw------- 1 root root    558524 Oct  6 04:39 dev.tsv
-rw------- 1 root root 414160198 Oct  6 04:40 final-model.pt
-rw------- 1 root root      1317 Oct  6 04:35 loss.tsv
-rw------- 1 root root 414160232 Oct  6 04:36 model_epoch_1.pt
-rw------- 1 root root 414160232 Oct  6 04:38 model_epoch_2.pt
-rw------- 1 root root 414160232 Oct  6 04:39 model_epoch_3.pt
-rw------- 1 root root    506400 Oct  6 04:40 test.tsv
-rw------- 1 root root     10412 Oct  6 04:40 training.log


What are these files?
* ```best-model.pt```: the best performing model encountered during training
* ```dev.tsv```: space-separated token, ground truth label, and predicted label for the dev.txt file
* ```final-model.pt```: The model as it stood when training ended
* ```loss.tsv```: A truly tab-separated file with the following columns:
** EPOCH
** TIMESTAMP
** BAD_EPOCHS
** LEARNING_RATE
** TRAIN_LOSS
** DEV_LOSS
** DEV_PRECISION
** DEV_RECALL
** DEV_F1
** DEV_ACCURACY
* ```model_epoch_##.pt```: The model produced at the end of each epoch
* ```test.tsv```: Same as dev.tsv but for test data
* ```training.log```: Same as the output of the notebook cell
Note that most of the ".tsv" files are not really tab-separated; they're white space delimited.
* ```weights.txt```: Model weights (must be specifically requested at training time; we won't use them).

## Tagging Text

We can tag any text we'd like using one of the models we've built.  First, we load some packages.

In [15]:
from flair.data import Sentence
from flair.models import SequenceTagger

Now we'll load a saved model from disk.

In [16]:
model = SequenceTagger.load(f'{labdir}/resources/taggers/{runname}/final-model.pt')
print(model)

2025-10-06 04:40:17,688 SequenceTagger predicts: Dictionary with 19 tags: O, S-LOC, B-LOC, E-LOC, I-LOC, S-PER, B-PER, E-PER, I-PER, S-ORG, B-ORG, E-ORG, I-ORG, S-MISC, B-MISC, E-MISC, I-MISC, <START>, <STOP>
SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      '0-/root/.flair/embeddings/glove.gensim'
      (embedding): Embedding(400001, 100)
    )
    (list_embedding_1): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
    (list_embedding_2): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=4196, out_features=4196, bias=True)
  (rnn): LSTM(4196, 256, batch_first=True, bidirect

We're ready to tag some data. Let's create a sample sentence, then tag it.

In [17]:
from textwrap import wrap

# create example sentence
sentence = Sentence("The Washington Post reported that Washington DC team the" +
        " Washington Commanders visited George Washington's Mt. Vernon estate.")
# Use wrap to make sure output doesn't run off page
print('\n'.join(wrap(sentence.to_plain_string())), '\n')

# predict the tags
model.predict(sentence)
print('\n'.join(wrap(sentence.to_tagged_string())))

The Washington Post reported that Washington DC team the Washington
Commanders visited George Washington's Mt. Vernon estate. 

Sentence[19]: "The Washington Post reported that Washington DC team
the Washington Commanders visited George Washington's Mt. Vernon
estate." → ["Washington Post"/ORG, "Washington DC"/ORG, "Washington
Commanders"/ORG, "George Washington"/PER, "Mt. Vernon"/LOC]


## Assignment

A. (10 points) Train a named entity recognizer as described above on the ```train.txt``` partition of the CoNLL 2003 English NER data.
This will be your 'base run.'
The system will report dev scores at each iteration.
Choose the number of epochs you will use for all of your runs.
Within 20 iterations you should see F1 scores on the dev partition of 95% or better.
You may choose a higher number if you want to see how good your system can get.
You may choose a lower number if you are running out of resources, but be sure to run at least ten epochs.
Report overall precision, recall, and F1 on the test set, as well as precision, recall, and F1 for each of the identified entity types.
Create a plot of the (decreasing) loss values from the training process,
where the x-axis is epoch and the y-axis is loss.
Then, train the system three more times on the same test data.
The purpose here is to see the the variation in scores caused by the randomness inherent in training a neural system.
Report your minimum, mean (of the four values),
and maximum overall F1 scores on the test data.
Plot the F1 performance on both the dev and test data,
where the x-axis is epoch and the y-axis is F1.
You will find all of these data in ```loss.tsv```.

B. (10 points) Examine how performance improves as more training data is used.
The training file has about 220K lines.
First, divide the data into 25K line chunks (preferably splitting between sentences).
[Note: you might be able to use FLAIR's ```Corpus.downsample``` for this, although I have not tried that.]
Then incrementally, train a system on the first chunk, the first two chunks, the first three chunks, etc.
Run each on the same number of iterations as your base run.
Plot the F1 learning curve as additional training data is made available,
where the x-axis is the amount of training data and the y-axis is F1.

C. (10 points) Study how performance changes when we don’t care about entity types.
First, create copies of both the training and the test data and map all entity types in the copies onto a single entity type, ENT
(i.e., map B-PER to B-ENT, I-PER to I-ENT, etc.).
You will end up with just three tags: B-ENT, I-ENT, and O.
Retrain the system on the new data, using the same number of iterations as you did in your first training.
Compare the scores in overall precision, recall, and F1, when type information is included
(using your base run) against this new run that maps all types to ENT.
Describe why you think there is/is not a difference.

D. (10 points) Explore how the system works on examples of your own selection.
First, create a new test file containing at least ten sentences.
You should include at least ten named entities of each of the three types PER, ORG, and LOC.
Your examples should include entities you think will be easy to identify, as well as a variety of difficult cases.
Run the tagger from the best saved model on your test examples.

Include a list of your sentences and the named entities the system found in your sentences in your pdf submission.
Indicate whether the system failed on any of your easy examples, and describe how it fared on your difficult examples.

## Submissions

As usual, your submission is to be a single pdf file.
Please do not submit a huge file.
There is no need to include outputs beyond what is requested.
If your pdf is longer than twenty or thirty pages,
you need to cut something out.
Remember that Colab/Jupyter notebooks have a tendency to truncate outputs over a certain width or length.
Please check your pdf submission to make sure that all the output you want to appear in it is actually there and has not run off the edge of the page.